# t-SNE and UMAP

PCA can only draw straight projections. Digits, cells, word vectors and many other datasets lie on curved surfaces in high dimensions, and a flat shadow of them overlaps itself. **t-SNE** and **UMAP** build a two-dimensional map that keeps *neighbourhoods* intact: points that are close in the original space should be close on the page. They make good pictures. They are also easy to misread, so half of this notebook is about what the pictures do not say.

Order: intuition, maths, cost function, from-scratch version, library version, tuning, what breaks.

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import umap
from scipy.spatial.distance import pdist, squareform
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE, trustworthiness

from common.datasets import load_digits_data

warnings.filterwarnings("ignore", message="n_jobs value", category=UserWarning)
warnings.filterwarnings("ignore", message="IProgress not found")

SEED = 42
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=3, suppress=True)
plt.rcParams["figure.figsize"] = (6, 3.5)

Xd, yd = load_digits_data(n_samples=1000, seed=SEED)  # subsample so the notebook stays fast
X, y = Xd.to_numpy() / 16.0, yd.to_numpy()


def show(ax, emb, labels, title):
    ax.scatter(emb[:, 0], emb[:, 1], c=labels, cmap="tab10", s=7)
    ax.set_title(title, fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])

## 1. Intuition

Think of a social map of a party. Each person stands close to the people they talk to most. You do not care about the exact distance between two people across the room. You care that friends stay together and strangers stay apart. t-SNE and UMAP draw that kind of map, from 64 pixel values per digit down to two coordinates.

The consequence is important. These methods keep **who is near whom**. They do not keep how far apart groups are, how big groups are, or which direction is which. Section 7 shows each of these being misleading.

Start with the plain comparison on digits: PCA, t-SNE and UMAP side by side.

In [ ]:
t0 = time.time()
emb_pca = PCA(2, random_state=SEED).fit_transform(X)
emb_tsne = TSNE(2, perplexity=30, init="pca", random_state=SEED).fit_transform(X)
t1 = time.time()
emb_umap = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=SEED).fit_transform(X)
t2 = time.time()
fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))
for ax, (emb, name) in zip(axes, [(emb_pca, "PCA"), (emb_tsne, "t-SNE"), (emb_umap, "UMAP")]):
    show(ax, emb, y, name)
plt.show()
print(
    f"t-SNE took {t1 - t0:.1f}s including PCA, UMAP took {t2 - t1:.1f}s (the first UMAP run includes numba compilation)"
)

PCA shows overlapping blobs. t-SNE and UMAP pull the ten digits into ten distinct islands. That is the pitch for nonlinear methods. Now how each one works.

## 2. Maths: t-SNE

t-SNE turns "near" into probabilities, in both the original space and the map, and moves the map points until the two sets of probabilities agree.

**In the original space.** For point $i$, the probability that it would pick $j$ as a neighbour falls off as a Gaussian of distance:

$$p_{j\mid i}=\frac{\exp(-\lVert\mathbf x_i-\mathbf x_j\rVert^2/2\sigma_i^2)}{\sum_{k\ne i}\exp(-\lVert\mathbf x_i-\mathbf x_k\rVert^2/2\sigma_i^2)},\qquad p_{ij}=\frac{p_{j\mid i}+p_{i\mid j}}{2n}$$

Each point gets its own width $\sigma_i$, chosen so the **perplexity** hits a target you set. Perplexity is roughly the effective number of neighbours each point pays attention to. Dense regions get a small $\sigma_i$ and sparse ones a large $\sigma_i$, so each point sees about the same number of neighbours. That is why cluster density and size do not survive.

**In the map.** Use a Student-t distribution with one degree of freedom, which has heavier tails than a Gaussian:

$$q_{ij}=\frac{(1+\lVert\mathbf y_i-\mathbf y_j\rVert^2)^{-1}}{\sum_{k\ne l}(1+\lVert\mathbf y_k-\mathbf y_l\rVert^2)^{-1}}$$

The heavy tail matters. In two dimensions there is little room around a point, so moderately distant neighbours get pushed far away to make space (the "crowding problem"). A heavy-tailed $q$ lets them sit far apart at small cost.

**Cost function.** The Kullback-Leibler divergence between the two sets of probabilities:

$$C=\mathrm{KL}(P\parallel Q)=\sum_{i\ne j}p_{ij}\log\frac{p_{ij}}{q_{ij}}$$

It is minimised by gradient descent on the map coordinates. KL is lopsided. If $p_{ij}$ is large (close in the original) and $q_{ij}$ small (far on the map), the cost is large, so close pairs are kept together. If $p_{ij}$ is tiny and $q_{ij}$ is larger, the cost is small. So t-SNE guards local neighbourhoods hard and cares little about global layout. The gradient has an intuitive form: each pair is a spring that attracts when $p_{ij}>q_{ij}$ and repels when $p_{ij}<q_{ij}$.

$$\frac{\partial C}{\partial\mathbf y_i}=4\sum_j(p_{ij}-q_{ij})(\mathbf y_i-\mathbf y_j)(1+\lVert\mathbf y_i-\mathbf y_j\rVert^2)^{-1}$$

**Cost.** The exact method is $O(n^2)$ per step. Libraries use Barnes-Hut or FFT approximations, which makes tens of thousands of points practical and a million slow.

## 3. From scratch

A small version, to see the pieces: find each $\sigma_i$ by binary search to hit the perplexity, build $P$, then run plain gradient descent with momentum on the map. 300 points and a few hundred steps are enough to see the KL fall.

In [ ]:
def p_matrix(X: np.ndarray, perplexity: float = 30.0) -> np.ndarray:
    d2 = squareform(pdist(X, "sqeuclidean"))
    n = len(X)
    P = np.zeros((n, n))
    target = np.log(perplexity)  # entropy (in nats) that corresponds to the perplexity
    for i in range(n):
        lo, hi, beta = 1e-10, 1e10, 1.0  # beta = 1 / (2 sigma^2)
        di = np.delete(d2[i], i)
        for _ in range(60):
            p = np.exp(-(di - di.min()) * beta)
            p /= p.sum()
            entropy = -(p * np.log(p + 1e-12)).sum()
            if entropy > target:  # too flat, narrow the Gaussian
                lo, beta = beta, beta * 2 if hi == 1e10 else (beta + hi) / 2
            else:
                hi, beta = beta, beta / 2 if lo == 1e-10 else (beta + lo) / 2
        P[i, np.arange(n) != i] = p
    return (P + P.T) / (2 * n)


def q_matrix(Y: np.ndarray) -> np.ndarray:
    w = 1.0 / (1.0 + squareform(pdist(Y, "sqeuclidean")))
    np.fill_diagonal(w, 0.0)
    return w / w.sum(), w


def kl(P: np.ndarray, Q: np.ndarray) -> float:
    mask = P > 0
    return float((P[mask] * np.log(P[mask] / np.maximum(Q[mask], 1e-12))).sum())


sub = rng.choice(len(X), 300, replace=False)
Xs, ys = X[sub], y[sub]
P = p_matrix(Xs, perplexity=30)
print("P sums to", P.sum().round(3), " largest p_ij:", P.max().round(5))

Y = PCA(2, random_state=SEED).fit_transform(Xs)
Y = Y / Y.std() * 1e-2  # small start, as in the original method
vel = np.zeros_like(Y)
history = []
for step in range(500):
    Q, w = q_matrix(Y)
    grad = (
        4
        * ((P * 12 if step < 100 else P) - Q)[:, :, None]
        * (Y[:, None, :] - Y[None, :, :])
        * w[:, :, None]
    )
    vel = (0.5 if step < 100 else 0.8) * vel - 100.0 * grad.sum(axis=1)
    Y = Y + vel
    if step % 50 == 49:
        history.append((step + 1, kl(P, Q)))
print("KL(P||Q) by step:", [(s, round(v, 3)) for s, v in history])
fig, ax = plt.subplots(figsize=(5, 4.2))
show(ax, Y, ys, "from-scratch t-SNE, 300 digits")
plt.show()
print(
    "trustworthiness: from-scratch",
    round(trustworthiness(Xs, Y, n_neighbors=10), 3),
    " PCA",
    round(trustworthiness(Xs, PCA(2).fit_transform(Xs), n_neighbors=10), 3),
)

The first 100 steps use **early exaggeration** (multiplying $P$ by 12), which lets clusters move freely before they settle, a trick from the original paper. The KL ticks up when the exaggeration stops (the target $P$ shrinks), then falls from about 3.0 to 1.4 by step 500, and the trustworthiness printed above (0.88) beats PCA on the same points (0.83) using nothing but the neighbour probabilities. It is well short of the library's 0.99 in section 5, because 500 plain gradient steps on 300 points is a crude optimiser. A KL value is only comparable between maps of the same data and perplexity, so it is a tool for checking convergence, not for comparing methods.

## 4. Maths: UMAP

UMAP has the same shape as t-SNE with different pieces.

1. **Graph.** For each point find its `n_neighbors` nearest neighbours, then weight each edge by how close the neighbour is relative to the point's closest neighbour. Distances are rescaled per point, so a point in a sparse region still counts its neighbours as near. The result is a weighted graph of the data.
2. **Layout.** Start the map from a spectral embedding of that graph. Then move points so that connected pairs attract and, by sampling random pairs, unconnected ones repel. The cost is a cross-entropy between edge weights in the graph and in the map. The map's edge weight has a curve that is flat up to `min_dist` and then decays, so `min_dist` sets how tightly points may pack.

The differences that matter in practice:

| | t-SNE | UMAP |
|---|---|---|
| Neighbourhood size | `perplexity` (about 5 to 50) | `n_neighbors` (about 5 to 100) |
| Packing | none to set | `min_dist` (0 to 1) |
| Speed on large data | slower | faster |
| Global structure | weak | somewhat better, still weak |
| New points | cannot transform them | `transform` works |
| Cost | KL divergence | cross-entropy on graph edges |

"Somewhat better" is an empirical impression and depends on initialisation as much as on the method. Much of the global layout in either map comes from the initialisation (PCA for t-SNE here, spectral for UMAP by default) and not from the cost.

## 5. How well do they preserve neighbours?

**Trustworthiness** (in scikit-learn) measures how many of a point's nearest neighbours on the map really were among its nearest neighbours in the original space, with a penalty for intruders. 1 is perfect. It is a more honest measure than the look of a picture.

In [ ]:
for name, emb in [("PCA", emb_pca), ("t-SNE", emb_tsne), ("UMAP", emb_umap)]:
    print(
        f"{name:>6}: trustworthiness (10 neighbours) = {trustworthiness(X, emb, n_neighbors=10):.3f}"
    )

Both nonlinear maps keep neighbourhoods much better than PCA (0.99 and 0.99 against 0.82). This says nothing about global distances, which trustworthiness does not check.

## 6. Tuning

**Perplexity (t-SNE).** Small values look only at a handful of neighbours, so the map breaks into many small clumps. Large values blur local detail and the picture approaches a blob. Values between 5 and 50 are typical, and the number must be below the number of points.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, perp in zip(axes, (2, 10, 30, 100)):
    emb = TSNE(2, perplexity=perp, init="pca", random_state=SEED).fit_transform(X)
    tw = trustworthiness(X, emb, n_neighbors=10)
    print(f"perplexity {perp}: trustworthiness {tw:.3f}")
    show(ax, emb, y, f"perplexity {perp}: trust {tw:.3f}")
plt.show()

Trustworthiness is 0.975 at perplexity 2, 0.991 at 10 and 30, and 0.989 at 100, so the score moves little while the pictures change a lot. Very low perplexity breaks the classes into fragments and very high perplexity blurs local detail, and a score that stays flat does not tell you which picture to prefer. The default of 30 is a sound start. Always try at least two values, and distrust any structure that only appears at one.

**`n_neighbors` and `min_dist` (UMAP).** `n_neighbors` plays the role of perplexity: small values stress local detail and give fragments, large values stress broad structure. `min_dist` controls how tightly points pile up. Near 0 gives dense clumps that look like crisp clusters. Larger values spread points out and show internal structure.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for row, md in zip(axes, (0.0, 0.5)):
    for ax, nn in zip(row, (5, 15, 50)):
        emb = umap.UMAP(n_neighbors=nn, min_dist=md, random_state=SEED).fit_transform(X)
        show(ax, emb, y, f"n_neighbors={nn}, min_dist={md}")
plt.show()

## 7. What breaks: how to misread the plots

**Cluster sizes mean nothing.** t-SNE sets each point's bandwidth to see the same effective number of neighbours, so a dense cluster is expanded and a sparse one shrunk. Make three clusters in 10 dimensions with very different spreads (0.3, 1 and 3) and equal sizes and see what t-SNE draws.

In [ ]:
centres = np.eye(10)[:3] * 12
spreads = (0.3, 1.0, 3.0)
Xg = np.vstack([rng.normal(c, s, (150, 10)) for c, s in zip(centres, spreads)])
yg = np.repeat([0, 1, 2], 150)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
emb = TSNE(2, perplexity=30, init="pca", random_state=SEED).fit_transform(Xg)
show(axes[0], emb, yg, "t-SNE")
emb_u = umap.UMAP(random_state=SEED).fit_transform(Xg)
show(axes[1], emb_u, yg, "UMAP")
plt.show()
for j, s in enumerate(spreads):
    print(
        f"true spread {s}: t-SNE map std {emb[yg == j].std(axis=0).mean():.1f}, UMAP map std {emb_u[yg == j].std(axis=0).mean():.2f}"
    )

The true spreads differ tenfold. The map spreads printed above are the same to within a few per cent, in both methods. Do not conclude from a t-SNE plot that one group is more variable than another.

**Distances between clusters, and the layout, mean little.** What is next to what is partly an accident of the optimiser. Run t-SNE with three seeds and the arrangement of the digit islands changes.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.4))
for ax, seed in zip(axes, (0, 1, 2)):
    emb = TSNE(2, perplexity=30, init="random", random_state=seed).fit_transform(X)
    show(ax, emb, y, f"random init, seed {seed}")
plt.show()

The same ten islands appear, but their positions change. Which two islands touch is not reliable evidence of similarity. With `init="pca"` the layout is more stable, but that is the starting point's doing, not extra truth.

**Noise can look like clusters.** Pure Gaussian noise has no clusters at all.

In [ ]:
noise = rng.normal(size=(500, 20))
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, perp in zip(axes, (3, 30, 100)):
    emb = TSNE(2, perplexity=perp, init="pca", random_state=SEED).fit_transform(noise)
    ax.scatter(*emb.T, s=6)
    ax.set_title(f"500 random points, perplexity {perp}", fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])
plt.show()

At low perplexity the random cloud can develop lumps and gaps. A visible cluster is a hypothesis to test, not a finding.

**Other limits.**

- t-SNE has no `transform`: new points cannot be placed on an existing map without refitting. UMAP can transform new data, with some caveats.
- Both depend on the seed, so set `random_state`. That makes the result reproducible but also hides run-to-run variation, so check one or two other seeds.
- Scale and preprocess first, as with every distance-based method. Running PCA to 30 to 50 dimensions before t-SNE is standard for speed and noise.
- Do not run k-means or DBSCAN on a t-SNE map and take the groups as truth. The map has already distorted sizes and distances. Cluster in the original or PCA space, then use the map to look.

**Rule of thumb.** Use PCA to compress, denoise and as the first step of a pipeline. Use t-SNE or UMAP to look at the data and to check whether labels or clusters are plausible. Do not use them as the features of a model without a good reason.

Next, how to tell whether a clustering or a reduction is any good when there is no answer key.